# Chess Academy Pro — Voice Pack Generator

Generates HD voice packs using Kokoro TTS on a free GPU.

**Two modes:**
- **Full generation**: Generates all phrases for voices with no existing pack
- **Incremental update**: For voices with an existing `.bin` file uploaded, generates only NEW phrases and merges them in

**Instructions:**
1. Click **Runtime → Change runtime type → T4 GPU**
2. Click **Runtime → Run all**
3. When prompted, upload any existing `.bin` voice packs (or skip to generate fresh)
4. Wait for all cells to complete
5. Download the zip file from the last cell

In [ ]:
# Cell 1: Install dependencies
!pip install -q kokoro>=0.9.2 soundfile numpy
!apt-get -qq -y install espeak-ng > /dev/null 2>&1
print('Dependencies installed!')

In [ ]:
# Cell 2: Clone repo and load data
import json, struct, os, time, zipfile
import numpy as np
import soundfile as sf
from io import BytesIO

# Clone the repo to get repertoire.json
!git clone --depth 1 https://github.com/dyahnke-pro/chess-academy-pro.git repo 2>/dev/null || true

# Load repertoire data
for path in ['repo/src/data/repertoire.json', 'src/data/repertoire.json', '/content/repertoire.json']:
    if os.path.exists(path):
        with open(path, 'r') as f:
            repertoire = json.load(f)
        print(f'Loaded {len(repertoire)} openings from {path}')
        break
else:
    print('ERROR: Could not find repertoire.json!')
    print('Upload it manually (find it at src/data/repertoire.json in your project)')
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        repertoire = json.loads(uploaded[fname].decode())
        print(f'Loaded {len(repertoire)} openings from upload')

# Hash function — MUST match voiceService.ts hashText() exactly
def hash_text(text: str) -> str:
    h = 0
    for ch in text:
        code = ord(ch)
        h = ((h << 5) - h) + code
        h &= 0xFFFFFFFF
        if h >= 0x80000000:
            h -= 0x100000000
    return str(h)

In [ ]:
# Cell 3: Collect ALL phrases + identify which are new

def collect_all_phrases(repertoire):
    """Collect every phrase the app needs voiced."""
    phrases = set()

    # --- Repertoire phrases ---
    for opening in repertoire:
        if opening.get('overview'):
            phrases.add(opening['overview'])
        for idea in opening.get('keyIdeas', []):
            phrases.add(idea)
        for trap in opening.get('traps', []):
            phrases.add(trap)
        for warning in opening.get('warnings', []):
            phrases.add(warning)
        for v in opening.get('variations', []):
            if v.get('explanation'):
                phrases.add(v['explanation'].replace('*', ''))
            phrases.add(f"Well done! You've completed the {v['name']} line.")
            phrases.add(f"Line discovered! You've learned the {v['name']}.")
            phrases.add(f"Line perfected! You know the {v['name']} by heart.")
        for t in opening.get('trapLines', []):
            if t.get('explanation'):
                phrases.add(t['explanation'].replace('*', ''))
            phrases.add(f"Well done! You've completed the {t['name']} line.")
            phrases.add(f"Line discovered! You've learned the {t['name']}.")
        for w in opening.get('warningLines', []):
            if w.get('explanation'):
                phrases.add(w['explanation'].replace('*', ''))
            phrases.add(f"Well done! You've completed the {w['name']} line.")
            phrases.add(f"Line discovered! You've learned the {w['name']}.")
        phrases.add(f"Let's play the {opening['name']}. Remember your key ideas and play confidently.")

    for hint in ['Castle to safety.', 'Develop your knight.', 'Develop your bishop.',
                 'Bring your queen out.', 'Activate your rook.', 'Continue with the plan.']:
        phrases.add(hint)

    # --- Walkthrough annotation phrases (main lines + 201 sub-lines) ---
    ann_dir = None
    for candidate in ['repo/src/data/annotations', 'src/data/annotations', '/content/annotations']:
        if os.path.exists(candidate):
            ann_dir = candidate
            break

    ann_count = 0
    if ann_dir:
        for fname in sorted(os.listdir(ann_dir)):
            if not fname.endswith('.json'):
                continue
            with open(os.path.join(ann_dir, fname)) as f:
                ann_data = json.load(f)
            for m in ann_data.get('moveAnnotations', []):
                if m.get('annotation'):
                    phrases.add(m['annotation'])
                    ann_count += 1
            for sub in ann_data.get('subLines', []):
                if not isinstance(sub, dict):
                    continue
                for m in sub.get('moveAnnotations', []):
                    if m.get('annotation'):
                        phrases.add(m['annotation'])
                        ann_count += 1
        print(f'  Loaded {ann_count} walkthrough annotations from {ann_dir}')
    else:
        print('  WARNING: No annotations directory found!')

    return phrases

all_phrases = collect_all_phrases(repertoire)
all_phrases_sorted = sorted(all_phrases)
all_hashes = {hash_text(p) for p in all_phrases}
print(f'Total unique phrases needed: {len(all_phrases)}')

In [ ]:
# Cell 3.5: Upload existing voice packs (optional — skip to generate all fresh)
# Upload any .bin files you already have to avoid re-generating those clips.

def unpack_voice_pack(data):
    """Read existing .bin voice pack, return dict of {hash: audio_bytes}."""
    clips = {}
    offset = 0
    count = struct.unpack_from('<I', data, offset)[0]
    offset += 4
    for _ in range(count):
        hash_len = struct.unpack_from('<H', data, offset)[0]
        offset += 2
        hash_str = data[offset:offset + hash_len].decode('utf-8')
        offset += hash_len
        audio_len = struct.unpack_from('<I', data, offset)[0]
        offset += 4
        audio_data = data[offset:offset + audio_len]
        offset += audio_len
        clips[hash_str] = audio_data
    return clips

existing_packs = {}  # voice_id -> {hash: audio_bytes}

print('Upload existing .bin voice packs to skip re-generating those clips.')
print('(Just press Cancel/skip if you want to generate everything fresh)\n')

try:
    from google.colab import files
    uploaded = files.upload()
    for fname, data in uploaded.items():
        voice_id = fname.replace('.bin', '')
        clips = unpack_voice_pack(data)
        existing_packs[voice_id] = clips
        print(f'  ✅ {fname}: {len(clips)} existing clips loaded')
except KeyboardInterrupt:
    print('  Skipped — will generate all voices fresh.')
except Exception as e:
    print(f'  Upload skipped or failed: {e}')
    print('  Will generate all voices fresh.')

print(f'\nExisting packs loaded: {len(existing_packs)}')

In [ ]:
# Cell 4: Initialize Kokoro pipeline
from kokoro import KPipeline

# American English pipeline
pipeline_a = KPipeline(lang_code='a')
# British English pipeline
pipeline_b = KPipeline(lang_code='b')
print('Kokoro pipelines ready!')

In [ ]:
# Cell 6: Generate voice packs (incremental — skips existing clips)
VOICES = {
    'af_heart': 'a', 'af_bella': 'a', 'af_nicole': 'a', 'af_sarah': 'a', 'af_nova': 'a',
    'am_adam': 'a', 'am_eric': 'a', 'am_michael': 'a', 'am_liam': 'a',
    'bf_emma': 'b', 'bf_isabella': 'b',
    'bm_daniel': 'b', 'bm_george': 'b',
}

os.makedirs('voice_packs', exist_ok=True)

for voice_id, lang in VOICES.items():
    pipeline = pipeline_a if lang == 'a' else pipeline_b
    print(f'\n{"="*60}')
    print(f'Generating: {voice_id}')
    print(f'{"="*60}')

    # Load existing clips if we have them
    existing_clips = existing_packs.get(voice_id, {})
    if existing_clips:
        print(f'  Loaded {len(existing_clips)} existing clips — will skip those')

    # Figure out which phrases need generating
    phrases_to_generate = []
    for text in all_phrases_sorted:
        text_hash = hash_text(text)
        if text_hash not in existing_clips:
            phrases_to_generate.append((text, text_hash))

    print(f'  Phrases to generate: {len(phrases_to_generate)} (skipping {len(all_phrases_sorted) - len(phrases_to_generate)} existing)')

    clips = []  # New clips only
    start = time.time()
    errors = 0

    for i, (text, text_hash) in enumerate(phrases_to_generate):
        try:
            generator = pipeline(text, voice=voice_id, speed=1.0)
            all_audio = []
            for _, _, audio_chunk in generator:
                all_audio.append(audio_chunk)

            if all_audio:
                full_audio = np.concatenate(all_audio)
                wav_bytes = audio_to_mp3_bytes(full_audio, 24000)
                clips.append({'hash': text_hash, 'audio': wav_bytes})
        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f'  ERROR [{i+1}]: {str(e)[:80]}')

        if (i + 1) % 200 == 0 or i == len(phrases_to_generate) - 1:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed * 60 if elapsed > 0 else 0
            eta = (len(phrases_to_generate) - i - 1) / max(rate / 60, 0.01) / 60
            print(f'  [{i+1}/{len(phrases_to_generate)}] {elapsed:.0f}s, {rate:.0f}/min, ETA {eta:.1f}min, {errors} errors')

    # Merge: existing clips + new clips
    merged_clips = []
    for h, audio in existing_clips.items():
        merged_clips.append({'hash': h, 'audio': audio})
    merged_clips.extend(clips)

    # Pack and save
    packed = pack_voice_pack(merged_clips)
    out_path = f'voice_packs/{voice_id}.bin'
    with open(out_path, 'wb') as f:
        f.write(packed)
    size_mb = len(packed) / 1024 / 1024
    print(f'  Saved: {out_path} ({size_mb:.1f} MB, {len(merged_clips)} total clips [{len(existing_clips)} old + {len(clips)} new], {errors} errors)')

print('\nAll voice packs generated!')

In [ ]:
# Cell 6: Generate all voice packs
VOICES = {
    'af_heart': 'a', 'af_bella': 'a', 'af_nicole': 'a', 'af_sarah': 'a', 'af_nova': 'a',
    'am_adam': 'a', 'am_eric': 'a', 'am_michael': 'a', 'am_liam': 'a',
    'bf_emma': 'b', 'bf_isabella': 'b',
    'bm_daniel': 'b', 'bm_george': 'b',
}

os.makedirs('voice_packs', exist_ok=True)

for voice_id, lang in VOICES.items():
    pipeline = pipeline_a if lang == 'a' else pipeline_b
    print(f'\n{"="*60}')
    print(f'Generating: {voice_id}')
    print(f'{"="*60}')
    
    clips = []
    start = time.time()
    errors = 0
    
    for i, text in enumerate(phrases):
        text_hash = hash_text(text)
        try:
            # Generate audio
            generator = pipeline(text, voice=voice_id, speed=1.0)
            all_audio = []
            for _, _, audio_chunk in generator:
                all_audio.append(audio_chunk)
            
            if all_audio:
                full_audio = np.concatenate(all_audio)
                wav_bytes = audio_to_mp3_bytes(full_audio, 24000)
                clips.append({'hash': text_hash, 'audio': wav_bytes})
        except Exception as e:
            errors += 1
            if errors <= 5:
                print(f'  ERROR [{i+1}]: {str(e)[:80]}')
        
        if (i + 1) % 200 == 0 or i == len(phrases) - 1:
            elapsed = time.time() - start
            rate = (i + 1) / elapsed * 60
            eta = (len(phrases) - i - 1) / max(rate / 60, 0.01) / 60
            print(f'  [{i+1}/{len(phrases)}] {elapsed:.0f}s, {rate:.0f}/min, ETA {eta:.1f}min, {errors} errors')
    
    # Pack and save
    packed = pack_voice_pack(clips)
    out_path = f'voice_packs/{voice_id}.bin'
    with open(out_path, 'wb') as f:
        f.write(packed)
    size_mb = len(packed) / 1024 / 1024
    print(f'  Saved: {out_path} ({size_mb:.1f} MB, {len(clips)} clips, {errors} errors)')

print('\nAll voice packs generated!')

In [ ]:
# Cell 7: Zip and download
zip_path = 'voice_packs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir('voice_packs')):
        if fname.endswith('.bin'):
            fpath = os.path.join('voice_packs', fname)
            zf.write(fpath, fname)
            size_mb = os.path.getsize(fpath) / 1024 / 1024
            print(f'  Added {fname} ({size_mb:.1f} MB)')

zip_size = os.path.getsize(zip_path) / 1024 / 1024
print(f'\nZip created: {zip_path} ({zip_size:.1f} MB)')

# Auto-download in Colab
try:
    from google.colab import files
    files.download(zip_path)
    print('Download started!')
except:
    print(f'Not running in Colab — file saved to {zip_path}')